In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import tinycudann as tcnn
import time
import pickle
import sys
import os
sys.path.append('../')
from helpers import *

 
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
 
def get_rays(H, W, focal, c2w):
    """Generate ray origins and directions for a camera pose."""
    i, j = np.meshgrid(np.arange(W, dtype=np.float32),
                       np.arange(H, dtype=np.float32), indexing='xy')
    dirs = np.stack([(i - W * 0.5) / focal,
                     -(j - H * 0.5) / focal,
                     -np.ones_like(i)], -1)
    rays_d = np.sum(dirs[..., np.newaxis, :] * c2w[:3, :3], -1)
    rays_o = np.broadcast_to(c2w[:3, -1], rays_d.shape).copy()
    return rays_o, rays_d
 
 
def volume_render(raw, z_vals):
    rgb = torch.sigmoid(raw[..., :3])
    sigma_a = torch.relu(raw[..., 3])
    dists = torch.cat([z_vals[..., 1:] - z_vals[..., :-1],
                       torch.full_like(z_vals[..., :1], 1e10)], -1)
    alpha = 1.0 - torch.exp(-sigma_a * dists)
    trans = torch.clamp(1.0 - alpha + 1e-10, max=1.0)
    trans = torch.cat([torch.ones_like(trans[..., :1]), trans[..., :-1]], -1)
    weights = alpha * torch.cumprod(trans, -1)
    rgb_map = torch.sum(weights[..., None] * rgb, -2)
    depth_map = torch.sum(weights * z_vals, -1)
    acc_map = torch.sum(weights, -1)
    return rgb_map, depth_map, acc_map
 
 
def normalize_pts(pts, near, far):
    scene_range = far  # works for forward-facing / 360° object-centric scenes
    return (pts / scene_range) * 0.5 + 0.5  # map [-far, far] -> [0, 1]

filename = 'lego_400.npz'
if not os.path.exists(filename):
    !gdown --id 108jNfjPITTsTA0lE6Kpg7Ei53BUVL-4n # Lego

data = np.load(filename)
images = data['images']
poses = data['poses']
focal = data['focal']
H, W = images.shape[1:3]

images, val_images, test_images = np.split(images[...,:3], [100,107], axis=0)
poses, val_poses, test_poses = np.split(poses, [100,107], axis=0)

TEST_IDX = 10

print(val_images.shape, test_images.shape, focal)
plt.imshow(test_images[TEST_IDX,...])
plt.show()

In [ ]:
def hash_parameters(config):
  return {
        "encoding": {
            "otype": "Grid",
            "type": "Hash",
            "n_levels":          config[0],
            "n_features_per_level": 2,
            "log2_hashmap_size": config[1],
            "base_resolution":   config[2],
            "per_level_scale":   config[3],
        },
        "network": {
            "otype": "FullyFusedMLP",
            "activation": "ReLU",
            "output_activation": "None",
            "n_neurons": 64,
            "n_hidden_layers": 2,
        },
  }
 
def make_network_with_input_encoding(params, seed=0):
  torch.manual_seed(seed)
  np.random.seed(seed)
  return tcnn.NetworkWithInputEncoding(
      n_input_dims = 2,
      n_output_dims = 1,
      encoding_config = params["encoding"],
      network_config = params["network"]
  )

class Instant_NGP_NeRF(nn.Module):
    def __init__(self, num_layers=2, num_channels=64, params=None,
                 seed=0, near=2.0, far=6.0, device=device):
        super().__init__()
        torch.manual_seed(seed)
        self.near = near
        self.far = far
        self.encoding = tcnn.Encoding(
            3, params["encoding"], dtype=torch.float32
        ).to(device)
        layers = [nn.Linear(self.encoding.n_output_dims, num_channels), nn.ReLU()]
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(num_channels, num_channels))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(num_channels, 4))  # 3 rgb + 1 sigma
        self.network = nn.Sequential(*layers)
 
    def forward(self, pts):
        shape = pts.shape[:-1]
        flat = pts.reshape(-1, 3)
        normed = normalize_pts(flat, self.near, self.far)
        normed = normed.clamp(0.0, 1.0)  # safety clamp for tcnn
        out = self.network(self.encoding(normed))
        return out.reshape(*shape, 4)
    
def fit_instant_ngp_nerf(images, poses, focal, H, W,
                         val_images, val_poses,
                         model_config,
                         iters=50000, learning_rate=1e-2,
                         batch_size=1024, N_samples=128,
                         near=2.0, far=6.0, stratified=True,
                         num_layers=2, num_channels=64,
                         log_interval=5000, seed=0,
                         device=device, count_params=False):
    # ── Pre-compute and shuffle training rays ────────────────────────────────
    all_rays_o, all_rays_d, all_rgb = [], [], []
    for i in range(poses.shape[0]):
        ro, rd = get_rays(H, W, focal, poses[i])
        all_rays_o.append(ro.reshape(-1, 3))
        all_rays_d.append(rd.reshape(-1, 3))
        all_rgb.append(images[i].reshape(-1, 3))
    all_rays_o = np.concatenate(all_rays_o, 0)
    all_rays_d = np.concatenate(all_rays_d, 0)
    all_rgb    = np.concatenate(all_rgb, 0)
 
    rng = np.random.RandomState(seed)
    perm = rng.permutation(all_rays_o.shape[0])
    all_rays_o, all_rays_d, all_rgb = all_rays_o[perm], all_rays_d[perm], all_rgb[perm]
    

    torch.manual_seed(seed)
    np.random.seed(seed)
 
    params = hash_parameters(model_config)
    model = Instant_NGP_NeRF(
        num_layers=num_layers,
        num_channels=num_channels,
        params=params,
        seed=seed,
        near=near,
        far=far,
        device=device,
    ).to(device)
 
    if count_params:
        n_params = sum(p.numel() for p in model.parameters())
        print(f'Number of parameters: {n_params}')
 
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, eps=1e-15)
    losses, xs = [], []
    best_loss = float('inf')
    b_i = 0
    t0 = time.time()
 
    for i in range(iters):
        # ---- mini-batch of rays ----
        if b_i + batch_size > all_rays_o.shape[0]:
            b_i = 0
        ro = torch.tensor(all_rays_o[b_i:b_i + batch_size], dtype=torch.float32, device=device)
        rd = torch.tensor(all_rays_d[b_i:b_i + batch_size], dtype=torch.float32, device=device)
        target = torch.tensor(all_rgb[b_i:b_i + batch_size], dtype=torch.float32, device=device)
        b_i += batch_size
 
        # ---- sample points along rays ----
        z_vals = torch.linspace(near, far, N_samples, device=device)
        if stratified:
            z_vals = z_vals + torch.rand(batch_size, N_samples, device=device) * (far - near) / N_samples
        else:
            z_vals = z_vals.unsqueeze(0).expand(batch_size, -1)
        pts = ro[:, None, :] + rd[:, None, :] * z_vals[:, :, None]  # [B, N_samples, 3]
 
        # ---- forward + volume rendering ----
        optimizer.zero_grad()
        raw = model(pts)  # [B, N_samples, 4]
        rgb_map, _, _ = volume_render(raw, z_vals)
 
        loss = torch.mean((rgb_map - target) ** 2)
        loss.backward()
        optimizer.step()
 
        losses.append(loss.item())
        xs.append(i)
        if loss.item() < best_loss:
            best_loss = loss.item()
 
        if (i + 1) % log_interval == 0:
            psnr = -10.0 * np.log10(loss.item())
            elapsed = (time.time() - t0) / 60.0
            print(f'Iteration {i+1}/{iters}, Loss: {loss.item():.3e}, '
                  f'PSNR: {psnr:.2f} dB, Time: {elapsed:.1f} min')
 
    # ---- render validation image ----
    model.eval()
    ro_v, rd_v = get_rays(H, W, focal, val_poses[0])
    ro_flat, rd_flat = ro_v.reshape(-1, 3), rd_v.reshape(-1, 3)
    rgb_chunks = []
    chunk = 512
    with torch.no_grad():
        for ci in range(0, ro_flat.shape[0], chunk):
            ro_c = torch.tensor(ro_flat[ci:ci + chunk], dtype=torch.float32, device=device)
            rd_c = torch.tensor(rd_flat[ci:ci + chunk], dtype=torch.float32, device=device)
            n = ro_c.shape[0]
            z = torch.linspace(near, far, N_samples, device=device).unsqueeze(0).expand(n, -1)
            pts_c = ro_c[:, None, :] + rd_c[:, None, :] * z[:, :, None]
            raw = model(pts_c)
            rgb_c, _, _ = volume_render(raw, z)
            rgb_chunks.append(rgb_c.cpu().numpy())
    pred_image = np.concatenate(rgb_chunks, 0).reshape(H, W, 3)
    val_loss = np.mean((pred_image - val_images[0]) ** 2)
    val_psnr = -10.0 * np.log10(val_loss)
    print(f'Val PSNR: {val_psnr:.2f} dB')
 
    return {
        'state': model.state_dict(),
        'pred': pred_image,
        'losses': losses,
        'xs': xs,
        'best_loss': best_loss,
        'val_psnr': val_psnr,
    }

In [ ]:
N_samples = 512
batch_size = 1024
num_layers, num_channels = 2, 128
near, far = 2., 6.

# varying hash_table_size
model_params = {
    'num_levels': 13,
    'hash_table_size': 0,
    'base_resolution': 16,
    'n_features_per_level': 2.0,
}
param_vals = [13, 16]

# varying num_levels
# model_params = {
#     'num_levels': 0,
#     'hash_table_size': 19,
#     'base_resolution': 16,
#     'n_features_per_level': 2.0,
# }
# param_vals = [13, 16]

if model_params['num_levels'] == 0:
    param_to_vary = 'num_levels'
else:    
    param_to_vary = 'hash_table_size'
outputs = {}
to_save_outputs = {}
for param_val in param_vals:
    model_params[param_to_vary] = param_val
    print(f'{param_to_vary}: {model_params[param_to_vary]}')
    model_config = (int(model_params['num_levels']), int(model_params['hash_table_size']), model_params['base_resolution'], model_params['n_features_per_level'])
    output = fit_instant_ngp_nerf(
        images, poses, focal, H, W,
        val_images, val_poses,
        model_config=model_config,
        iters=50000, learning_rate=1e-2,
        batch_size=batch_size, N_samples=N_samples,
        near=near, far=far,
        num_layers=num_layers, num_channels=num_channels,
        log_interval=5000, seed=0, count_params=True
    )
    outputs[f'{model_params[param_to_vary]}'] = output                        


In [ ]:
outputs_plot = {}
for param_val in outputs.keys():
    model_params[param_to_vary] = param_val
    model_config = (int(model_params['num_levels']), int(model_params['hash_table_size']), model_params['base_resolution'], model_params['n_features_per_level'])
    params = hash_parameters(model_config)
    model = Instant_NGP_NeRF(
        num_layers=num_layers,
        num_channels=num_channels,
        params=params,
        seed=0,
        near=near,
        far=far,
        device=device,
    ).to(device)
    model.load_state_dict(outputs[param_val]['state'])
    model.eval()
    
    chunk = 512
    with torch.no_grad():
        # Generate rays for this test pose
        ro_v, rd_v = get_rays(H, W, focal, test_poses[TEST_IDX])
        ro_flat = ro_v.reshape(-1, 3)
        rd_flat = rd_v.reshape(-1, 3)

        # Render in chunks
        rgb_chunks = []
        for ci in range(0, ro_flat.shape[0], chunk):
            ro_c = torch.tensor(ro_flat[ci:ci + chunk],
                                dtype=torch.float32, device=device)
            rd_c = torch.tensor(rd_flat[ci:ci + chunk],
                                dtype=torch.float32, device=device)
            n = ro_c.shape[0]
            z = torch.linspace(near, far, N_samples,
                                device=device).unsqueeze(0).expand(n, -1)
            pts_c = ro_c[:, None, :] + rd_c[:, None, :] * z[:, :, None]
            raw = model(pts_c)
            rgb_c, _, _ = volume_render(raw, z)
            rgb_chunks.append(rgb_c.cpu().numpy())

        rendered = np.concatenate(rgb_chunks, 0).reshape(H, W, 3)

        outputs_plot[param_val] = {}
        outputs_plot[param_val]['best_pred'] = rendered

with open(f"3d_nerf/ingp.pkl", "wb") as f:
    pickle.dump(outputs_plot, f)
plot_error_heatmaps(test_images[TEST_IDX,...], outputs_plot, model_name="Instant-NGP")